# 🐱‍🔮 Sudoku Master — Flujo Completo
Pipeline completo: YOLO → Perspectiva → OCR → CNN → Backtracking

## PASO 0 · Instalación e imports

In [ ]:
!pip install ultralytics easyocr -q

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import copy
from ultralytics import YOLO
import easyocr
from google.colab import files, drive
from google.colab.patches import cv2_imshow

## PASO 1 · Cargar modelos

In [ ]:
drive.mount('/content/drive')

modelo_yolo = YOLO('/content/drive/MyDrive/Sudoku/modelo_entrenado/sudoku/weights/best.pt')
reader = easyocr.Reader(['en'])
print('Modelos cargados ✓')

## PASO 2 · Subir imagen y detectar orientación + recuadro con YOLO

In [ ]:
def corregir_orientacion(img, modelo):
    """Prueba las 4 rotaciones y se queda con la que YOLO detecte con más confianza."""
    rotaciones = {
        0:   img,
        90:  cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE),
        180: cv2.rotate(img, cv2.ROTATE_180),
        270: cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    }

    mejor_conf = 0
    mejor_img  = img
    mejor_ang  = 0

    for angulo, rotada in rotaciones.items():
        result = modelo(rotada, verbose=False)
        if len(result[0].boxes) > 0:
            conf = float(result[0].boxes.conf[0])
            if conf > mejor_conf:
                mejor_conf = conf
                mejor_img  = rotada
                mejor_ang  = angulo

    print(f'✓ Mejor orientación: {mejor_ang}° (confianza: {mejor_conf:.2%})')
    return mejor_img

# Subir imagen
uploaded = files.upload()
nombre_imagen = list(uploaded.keys())[0]
imagen_original = cv2.imread(nombre_imagen)

# Corregir orientación
imagen = corregir_orientacion(imagen_original, modelo_yolo)

# YOLO detecta el recuadro
results = modelo_yolo(imagen)

if len(results[0].boxes) == 0:
    print('❌ No se detectó ningún recuadro de sudoku')
else:
    x1, y1, x2, y2 = map(int, results[0].boxes.xyxy[0])
    img_recortada = results[0].orig_img[y1:y2, x1:x2]
    print(f'✓ Recuadro detectado: {x1},{y1} → {x2},{y2}')
    cv2_imshow(img_recortada)

## PASO 3 · Corrección de perspectiva

In [ ]:
def corregir_perspectiva(img):
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gris, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contorno = max(contornos, key=cv2.contourArea)

    epsilon = 0.02 * cv2.arcLength(contorno, True)
    esquinas = cv2.approxPolyDP(contorno, epsilon, True)

    if len(esquinas) != 4:
        print('⚠️ No se encontraron 4 esquinas, usando imagen sin corregir')
        return img

    pts = esquinas.reshape(4, 2).astype('float32')
    rect = np.zeros((4, 2), dtype='float32')
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]

    lado = 450
    dst = np.array([[0,0],[lado,0],[lado,lado],[0,lado]], dtype='float32')
    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(img, M, (lado, lado))

img_corregida = corregir_perspectiva(img_recortada)
print('✓ Perspectiva corregida')
cv2_imshow(img_corregida)

## PASO 4 · Dividir en 81 celdas

In [ ]:
alto, ancho = img_corregida.shape[:2]
celda_alto  = alto // 9
celda_ancho = ancho // 9
MARGEN = 4

celdas = []
for fila in range(9):
    fila_celdas = []
    for col in range(9):
        y1c = fila * celda_alto + MARGEN
        y2c = y1c + celda_alto - MARGEN * 2
        x1c = col * celda_ancho + MARGEN
        x2c = x1c + celda_ancho - MARGEN * 2
        fila_celdas.append(img_corregida[y1c:y2c, x1c:x2c])
    celdas.append(fila_celdas)

fig, axes = plt.subplots(9, 9, figsize=(9, 9))
for fila in range(9):
    for col in range(9):
        celda_rgb = cv2.cvtColor(celdas[fila][col], cv2.COLOR_BGR2RGB)
        axes[fila][col].imshow(celda_rgb)
        axes[fila][col].axis('off')
plt.tight_layout()
plt.show()
print('✓ 81 celdas generadas')

## PASO 5 · Detectar dígitos con EasyOCR

In [ ]:
def predecir_celda(celda):
    gris = cv2.cvtColor(celda, cv2.COLOR_BGR2GRAY)
    gris = cv2.resize(gris, (gris.shape[1]*3, gris.shape[0]*3), interpolation=cv2.INTER_CUBIC)
    resultado = reader.readtext(gris, allowlist='123456789', detail=1)
    if not resultado:
        return 0
    texto     = resultado[0][1]
    confianza = resultado[0][2]
    if texto.isdigit() and 1 <= int(texto) <= 9:
        return int(texto)
    return 0

sudoku = []
for fila in range(9):
    fila_nums = []
    for col in range(9):
        digito = predecir_celda(celdas[fila][col])
        fila_nums.append(digito)
    sudoku.append(fila_nums)

print('Sudoku detectado:')
print('─' * 25)
for i, fila in enumerate(sudoku):
    if i % 3 == 0 and i != 0:
        print('─' * 25)
    fila_str = ''
    for j, num in enumerate(fila):
        if j % 3 == 0 and j != 0:
            fila_str += '│ '
        fila_str += f'{num} '
    print(fila_str)

## PASO 6 · Entrenar modelo CNN con CSV (estratificado por dificultad)

In [ ]:
import pandas as pd
from tensorflow import keras

# Cargar CSV completo (solo columnas necesarias)
df_full = pd.read_csv('/content/drive/MyDrive/Sudoku/sudoku.csv')

# Calcular número de ceros (dificultad) de cada puzzle
df_full['n_ceros'] = df_full['puzzle'].apply(lambda x: x.count('0'))

# Estratificar: muestrear igual cantidad de cada nivel de dificultad
# Agrupamos en 5 niveles de dificultad
df_full['nivel'] = pd.cut(df_full['n_ceros'], bins=5, labels=[1,2,3,4,5])

MUESTRAS_POR_NIVEL = 20000  # 20k por nivel = 100k total
df = df_full.groupby('nivel', group_keys=False).apply(
    lambda x: x.sample(min(len(x), MUESTRAS_POR_NIVEL), random_state=42)
).reset_index(drop=True)

print(f'✓ Dataset estratificado: {len(df)} registros')
print(df.groupby('nivel')['n_ceros'].agg(['count','min','max']))

X = np.array(df['puzzle'].apply(lambda x: [int(c) for c in x]).tolist()) / 9.0
y = np.array(df['solution'].apply(lambda x: [int(c) for c in x]).tolist()) - 1

print(f'✓ X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
modelo_sudoku = keras.Sequential([
    keras.layers.Reshape((9, 9, 1), input_shape=(81,)),
    keras.layers.Conv2D(64,  (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(64,  (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(256, (1,1), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(9,   (1,1), activation='softmax', padding='same'),
    keras.layers.Reshape((81, 9))
])

modelo_sudoku.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

modelo_sudoku.fit(X.reshape(-1, 81), y, epochs=20, batch_size=64, validation_split=0.1)

modelo_sudoku.save('modelo_sudoku.keras')
!cp modelo_sudoku.keras '/content/drive/MyDrive/Sudoku/'
print('✓ Modelo guardado en Drive')

## PASO 7 · Resolver el sudoku
Elige uno de los tres métodos:

### Opción A — Solo backtracking (100% fiable)

In [ ]:
def es_valido(t, fila, col, num):
    if num in t[fila]: return False
    if num in [t[i][col] for i in range(9)]: return False
    bf, bc = (fila//3)*3, (col//3)*3
    for i in range(bf, bf+3):
        for j in range(bc, bc+3):
            if t[i][j] == num: return False
    return True

def backtrack(t):
    for i in range(9):
        for j in range(9):
            if t[i][j] == 0:
                for num in range(1, 10):
                    if es_valido(t, i, j, num):
                        t[i][j] = num
                        if backtrack(t): return True
                        t[i][j] = 0
                return False
    return True

tablero = copy.deepcopy(sudoku)
backtrack(tablero)

print('Sudoku resuelto (backtracking):')
print('─' * 25)
for i, fila in enumerate(tablero):
    if i % 3 == 0 and i != 0: print('─' * 25)
    fila_str = ''
    for j, num in enumerate(fila):
        if j % 3 == 0 and j != 0: fila_str += '│ '
        fila_str += f'{num} '
    print(fila_str)

### Opción B — Solo CNN (predicción pura)

In [ ]:
def resolver_con_modelo(sudoku, modelo):
    puzzle = np.array([n for fila in sudoku for n in fila]).reshape(1, 81) / 9.0
    pred   = modelo.predict(puzzle.reshape(-1, 81), verbose=0)
    solucion = np.argmax(pred[0], axis=1) + 1
    resultado = []
    for i in range(9):
        fila = []
        for j in range(9):
            if sudoku[i][j] != 0:
                fila.append(sudoku[i][j])
            else:
                fila.append(int(solucion[i*9 + j]))
        resultado.append(fila)
    return resultado

solucion = resolver_con_modelo(sudoku, modelo_sudoku)

print('Sudoku resuelto (CNN):')
print('─' * 25)
for i, fila in enumerate(solucion):
    if i % 3 == 0 and i != 0: print('─' * 25)
    fila_str = ''
    for j, num in enumerate(fila):
        if j % 3 == 0 and j != 0: fila_str += '│ '
        fila_str += f'{num} '
    print(fila_str)

### Opción C — CNN + Backtracking (IA predice, programación valida) ⭐ RECOMENDADO

In [ ]:
# CNN sugiere la solución
puzzle = np.array([n for fila in sudoku for n in fila]).reshape(1, 81) / 9.0
pred   = modelo_sudoku.predict(puzzle.reshape(-1, 81), verbose=0)
solucion_modelo = np.argmax(pred[0], axis=1) + 1

tablero = []
for i in range(9):
    fila = []
    for j in range(9):
        if sudoku[i][j] != 0:
            fila.append(sudoku[i][j])
        else:
            fila.append(int(solucion_modelo[i*9 + j]))
    tablero.append(fila)

# Backtracking valida y corrige
def es_valido(t, fila, col, num):
    if num in t[fila]: return False
    if num in [t[i][col] for i in range(9)]: return False
    bf, bc = (fila//3)*3, (col//3)*3
    for i in range(bf, bf+3):
        for j in range(bc, bc+3):
            if t[i][j] == num: return False
    return True

def backtrack(t):
    for i in range(9):
        for j in range(9):
            if sudoku[i][j] == 0:
                if not es_valido(t, i, j, t[i][j]):
                    t[i][j] = 0
                    for num in range(1, 10):
                        if es_valido(t, i, j, num):
                            t[i][j] = num
                            if backtrack(t): return True
                            t[i][j] = 0
                    return False
    return True

backtrack(tablero)

print('Sudoku resuelto (CNN + Backtracking):')
print('─' * 25)
for i, fila in enumerate(tablero):
    if i % 3 == 0 and i != 0: print('─' * 25)
    fila_str = ''
    for j, num in enumerate(fila):
        if j % 3 == 0 and j != 0: fila_str += '│ '
        fila_str += f'{num} '
    print(fila_str)

## UTILIDADES · Descargar modelos

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/Sudoku/modelo_entrenado/sudoku/weights/best.pt')
files.download('/content/drive/MyDrive/Sudoku/modelo_sudoku.keras')